In [8]:
%pip install -r requirements-process.txt

import torchvision.transforms as transforms
from PIL import Image
from minio import Minio
import webdataset as wds
import io
import os

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.gif', '.bmp')

client = Minio(
    os.environ.get('AWS_S3_ENDPOINT'), 
    access_key = os.environ.get('AWS_ACCESS_KEY_ID'), 
    secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
)
bucket_name = os.environ.get('AWS_S3_BUCKET')

preproc = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

def download_image(client, object_name):
    try:
        response = client.get_object(bucket_name, object_name)
        image_data = response.read(decode_content=True)
    finally:
        if response:
            response.close()
            response.release_conn()
    return(Image.open(io.BytesIO(image_data)))


def download_text(client, object_name):
    try:
        response = client.get_object(bucket_name, object_name)
        text_data = response.read(decode_content=True)
    finally:
        if response:
            response.close()
            response.release_conn()
    return(text_data)


def preprocess(image):
    return preproc(image)

def upload_shard(filename):
    print(filename)

objects = client.list_objects(bucket_name=bucket_name, 
                              prefix="dataset/")
sink = wds.ShardWriter("samples-%02d.tar", post=upload_shard)

index = 0
for obj in objects:
    if obj.object_name.lower().endswith(IMAGE_EXTENSIONS):
        image_data = download_image(client, obj.object_name)
        processed_image = transforms.ToPILImage()(preprocess(image_data))
        image_byte_array = io.BytesIO()
        processed_image.save(image_byte_array, format='PNG')
        image_byte_array = image_byte_array.getvalue()
        txt_object_name = obj.object_name.rsplit('.', 1)[0] + ".txt"
        try:
            txt_data = download_text(client, txt_object_name)

            # ensure the text data is in bytes format
            if isinstance(txt_data, str):
                txt_data = txt_data.encode('utf-8')
        except Exception:
            # If the text file doesn't exist, default to empty bytes
            print(f"Warning: No text file found for {obj.object_name}, using empty string.")
            txt_data = b""

        # --- Write Sample ---
        sample = {
            "__key__": "%06d" % index,
            "png": image_byte_array,
            "txt": txt_data  # Replaced "abc".encode() with downloaded txt_data
        }
        sink.write(sample)
        index += 1
sink.close()



Note: you may need to restart the kernel to use updated packages.
# writing samples-00.tar 0 0.0 GB 0
samples-00.tar
